# 05D 真实 COF 机器学习数据集：从公开数据到可复现 baseline

这一章不再使用人工 target。我们直接读取已经公开发表的 COF 数据，练习 **真实 feature + 真实模拟 target + provenance + baseline**。

本章使用/介绍三类互补数据源：

1. **COFSpace / CoRE COF**：1060 个 CoRE COF 的多气体 GCMC adsorption 数据和结构/化学/能量特征，适合标准监督学习。
2. **CURATED-COFs adsorption data**：实验报道 COF 对应的计算吸附性质，可通过 COF ID 与结构库连接，适合学习 `CIF ↔ property table`。
3. **ReDD-COFFEE CO₂ capture HTS**：大规模 hypothetical COF 的 feature、GCMC target、固定 train/test split 和 SHAP 工作流，适合学习 high-throughput screening。

> 这里的“真实 target”指论文公开的分子模拟/数据库结果，而不是实验测量值。训练前必须保留温度、压力、气体、单位和计算协议。

## 1. Dataset A — COFSpace：1060 CoRE COFs 的 CO₂ adsorption

COFSpace 公开了用于论文机器学习的 feature tables 与 gas-uptake data。下面直接读取 **CO₂, 1 bar** 的训练表。它包含 PLD、LCD、accessible surface area、porosity、Henry coefficient、元素比例和 CO₂ uptake。

In [ ]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv'
cofspace = pd.read_csv(url)
print(cofspace.shape)
display(cofspace.head())
print(cofspace.dtypes)

### 第一个真实 adsorption baseline

我们暂时不追求论文级最优模型，只验证完整流程。注意：`KCO2` 是能量/吸附亲和力相关特征，是否使用它取决于你的科学问题。若目标是只从廉价结构特征预测 uptake，可以把它排除；若目标是复现论文中的 surrogate setting，则可以保留。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

target = 'CO2-1 bar (mol/kg)'
features = [c for c in cofspace.columns if c != target]
X = cofspace[features]
y = cofspace[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, pred))
print('R2 :', r2_score(y_test, pred))
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
display(importance.to_frame('importance'))

## 2. Dataset B — CURATED-COFs adsorption properties

`nachatz/cof-data` 整理了来自 CURATED-COFs / Materials Cloud 的 adsorption tasks。`properties.csv` 包含 H₂、O₂、CO₂、CH₄、N₂、Xe、Kr、H₂O、H₂S 等性质；`simple_features.csv` 提供 ASA、density 和 largest sphere 等简单结构特征。两个表通过 COF ID 连接。

这一部分特别适合练习：**不同来源的 feature table 和 property table 如何通过稳定 ID merge。**

In [ ]:
prop_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/properties.csv'
feat_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/simple_features.csv'
properties = pd.read_csv(prop_url)
simple_features = pd.read_csv(feat_url)
dataset_b = simple_features.merge(properties, left_on='cof', right_on='name', how='inner')
print('features:', simple_features.shape, 'properties:', properties.shape, 'merged:', dataset_b.shape)
display(dataset_b[['cof','ASA_m^2/g','Density','LS','co2_30bar','co2_ads_unit','co2_henry','co2_henry_unit']].head())

In [ ]:
# 只用三个廉价结构特征预测 CO2 30 bar uptake：这是 deliberately simple baseline
df = dataset_b[['ASA_m^2/g','Density','LS','co2_30bar']].dropna()
X = df[['ASA_m^2/g','Density','LS']]
y = df['co2_30bar']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
m2 = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
p2 = m2.predict(X_test)
print('n =', len(df), 'MAE =', mean_absolute_error(y_test, p2), 'R2 =', r2_score(y_test, p2))

## 3. Dataset C — ReDD-COFFEE CO₂ capture high-throughput screening

`SupportingInformation_CO2captureHTS_2024` 是更接近真实科研工程的案例：

- `features.csv`：大规模 COF descriptors；
- `results.csv`：GCMC / screening targets；
- `structs_train.txt` / `structs_test.txt`：固定数据划分；
- ML scripts：feature reduction、模型训练、prediction 和 SHAP。

完整 feature archive 较大，因此不建议每个 Colab runtime 自动下载全部数据。课程采用两层方式：前两节用小型真实表直接运行；这一节阅读并复现其 **data contract**，需要做大规模项目时再下载完整 ReDD-COFFEE 数据。

官方公开仓库：https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024

In [ ]:
# results.csv 约数 MB，可直接检查；完整 features archive 更大。
results_url = 'https://raw.githubusercontent.com/jsdvos/SupportingInformation_CO2captureHTS_2024/master/Step2_MachineLearning/data/input/results.csv'
redd_results = pd.read_csv(results_url, sep=';')
print(redd_results.shape)
display(redd_results.head())
print(list(redd_results.columns))

## 4. 三个数据集分别教什么？

| Dataset | 规模/类型 | 适合教学的问题 |
|---|---|---|
| COFSpace / CoRE COF | ~10³，真实模拟 label | 标准 supervised regression、feature importance、多气体/多压力 target |
| CURATED-COFs adsorption | 实验 COF + 计算性质 | CIF/ID/property merge、单位与条件、small-data baseline |
| ReDD-COFFEE HTS | ~10⁵ hypothetical COFs | fixed split、feature reduction、surrogate screening、SHAP、scale |

不要把三个数据集直接拼成一个训练表。它们的结构来源、计算协议、特征定义和 target 条件不同。正确做法是先分别建立 dataset card，再判断是否可以做 transfer learning、external validation 或 domain-shift study。

## 5. 建议练习

1. 在 COFSpace 中比较 0.1、1、5、10 bar 的 CO₂ 模型，观察 feature importance 是否随压力变化。
2. 在 CURATED 数据中分别预测 `co2_30bar` 与 `co2_henry`，解释为什么控制因素可能不同。
3. 将 05B 从 CIF 计算出的 composition descriptors 与 Dataset B 的 property table 通过 COF ID 合并。
4. 对 ReDD-COFFEE 使用作者提供的固定 train/test list，而不是重新随机切分，并比较结果。
5. 为每个实验写出 dataset card：source、structure type、target、T/P、unit、simulation method、features、split、license/citation。

## 数据来源与引用

- COFSpace: G. Onder Aksu et al., *The COF Space: Materials Features, Gas Adsorption, and Separation Performances Assessed by Machine Learning*, ACS Materials Letters. Data and scripts: https://github.com/gokhanonderaksu/COFSpace
- CURATED-COFs / Materials Cloud: D. Ongari et al., *Building a consistent and reproducible database for adsorption evaluation in Covalent-Organic Frameworks*. Data DOI: 10.24435/materialscloud:z6-jn.
- ReDD-COFFEE / CO₂ capture HTS: https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024

使用这些数据发表研究时，请引用原始论文和数据记录，而不是只引用本教程。